## Fase 1: Preparação e Serialização do Modelo (Pipeline Offline)

## Relatório de Métricas do modelo 

Este documento descreve os metadados, as variáveis de entrada e os resultados de avaliação registrados em `models/v1/metricas_v1.json`.

## Identificação e treinamento

| Campo | Valor |
|---|---|
| Versão | `v1` |
| Modelo salvo | Gradient Boosting |
| Data de treinamento registrada | 16/07/2026 |
| Dados usados no modelo final | 100% da base (treino + teste) |
| Etapa do retreino final | Fase 6a |

O modelo salvo foi retreinado com toda a base após a avaliação da Fase 5. Por isso, as métricas de teste abaixo não são métricas calculadas sobre o modelo final aplicado à base completa: elas vêm do modelo avaliado no split 80/20 da Fase 5, que preserva uma parcela de dados não vista durante o treinamento e fornece uma estimativa mais honesta do desempenho.

O `StandardScaler` também foi reajustado com 100% da base para o modelo final salvo. As métricas apresentadas, porém, correspondem à avaliação anterior no split 80/20.

## Variáveis utilizadas

O modelo utiliza 23 features:

| Grupo | Features |
|---|---|
| Características do imóvel | `bedrooms`, `bathrooms`, `sqft_living`, `sqft_lot`, `floors`, `waterfront`, `view`, `condition`, `grade`, `sqft_basement` |
| Idade e reforma | `idade_imovel`, `foi_reformado` |
| Localização | `lat`, `long` |
| Faixas de CEP codificadas | `zipcode_faixa_faixa_1`, `zipcode_faixa_faixa_2`, `zipcode_faixa_faixa_3`, `zipcode_faixa_faixa_4`, `zipcode_faixa_faixa_5`, `zipcode_faixa_faixa_6`, `zipcode_faixa_faixa_7`, `zipcode_faixa_faixa_8`, `zipcode_faixa_faixa_9` |

Os nomes e a quantidade de features foram transcritos do arquivo de métricas; a unidade e a regra de cálculo de cada variável não estão especificadas nele.

## Resultados da avaliação

Os valores estão arredondados para duas casas decimais, quando aplicável.

| Conjunto | MAE | MSE | RMSE | R² |
|---|---:|---:|---:|---:|
| Treino | 104.550,97 | 30.302.296.089,05 | 174.075,55 | 0,7681 |
| Teste (split 80/20) | 77.466,75 | 20.283.292.615,64 | 142.419,42 | 0,8658 |

### Como interpretar

- **MAE**: erro absoluto médio entre as previsões e os valores observados. É expresso na mesma unidade da variável prevista.
- **MSE**: média dos erros ao quadrado. Penaliza erros maiores e fica na unidade da variável prevista ao quadrado.
- **RMSE**: raiz do MSE; é expresso na mesma unidade da variável prevista e dá mais peso a erros grandes que o MAE.
- **R²**: proporção da variação dos valores observados explicada pelo modelo. Quanto mais próximo de 1, maior a proporção explicada na avaliação.

O conjunto de teste apresentou MAE e RMSE menores e R² maior que o conjunto de treino. Esses resultados devem ser lidos conforme o processo de avaliação registrado; não significam que o modelo final retreinado tenha sido medido novamente em dados independentes.

## Fonte

Todos os dados deste documento foram extraídos de `models/v1/metricas_v1.json`. 

## Fase 2: Estrutura Modular da API FastAPI

A aplicação é organizada em camadas para separar validação, inferência, rotas e inicialização.

- **`schemas/` ou `schemas.py`** — define os modelos Pydantic (`BaseModel` e `Field`) para validar o JSON recebido e estruturar as respostas.
- **`services/` ou `services.py`** — carrega o modelo serializado e executa a inferência, mantendo as regras de negócio independentes do protocolo HTTP.
- **`routers/` ou `routers.py`** — agrupa os endpoints de negócio usando `APIRouter`.
- **`main.py`** — ponto de entrada da aplicação; cria a instância FastAPI, inclui os roteadores e gerencia o ciclo de vida.

## Fase 3: Ciclo de Vida e Inicialização Otimizada (Lifespan)

O ciclo de vida da API é gerenciado pelo `lifespan` do FastAPI, definido em `main.py`.

### Inicialização

Durante o startup, `initialize_model()` carrega e valida o modelo e o scaler. Os artefatos ficam em memória antes de a aplicação começar a receber requisições.

### Processamento das previsões

A rota em `routers.py` chama o serviço em `services.py`, que usa o modelo e o scaler já carregados.

> **Importante:** a rota de predição não lê nem recarrega arquivos `.pkl` ou `.joblib` a cada requisição. Se o serviço for chamado sem que os artefatos tenham sido inicializados, ele gera um erro explícito em vez de tentar carregá-los do disco.

### Encerramento

Quando a aplicação é encerrada, o `lifespan` limpa as referências aos artefatos em memória.